# SwitchLingua Phase 1
## 1. Dataset Exploration and Data Preparation

### Step 1: Dataset exploration

In [42]:
from datasets import load_dataset
from itertools import islice
import ast
import pandas as pd

In [43]:
# Load the dataset in streaming mode
dataset = load_dataset(
    "Shelton1013/SwitchLingua_text",
    split="train",
    streaming=True
)

print(dataset)

IterableDataset({
    features: ['topic', 'tense', 'perspective', 'cs_ratio', 'gender', 'age', 'education_level', 'first_language', 'second_language', 'conversation_type', 'cs_function', 'cs_type', 'data_generation_result', 'fluency_result', 'naturalness_result', 'cs_ratio_result', 'social_cultural_result', 'summary', 'score'],
    num_shards: 15
})


In [44]:
# Use the first 1,000 streamed rows for a quick parsing/format check.
# This sample is not intended to represent the full language-pair distribution.
sample_rows = list(islice(dataset, 1000))

print("Sample size:", len(sample_rows))

Sample size: 1000


In [45]:
# Inspect the columns
print(sample_rows[0].keys())

dict_keys(['topic', 'tense', 'perspective', 'cs_ratio', 'gender', 'age', 'education_level', 'first_language', 'second_language', 'conversation_type', 'cs_function', 'cs_type', 'data_generation_result', 'fluency_result', 'naturalness_result', 'cs_ratio_result', 'social_cultural_result', 'summary', 'score'])


In [46]:
# Inspect one example
sample_rows[0]

{'topic': 'general',
 'tense': 'Present',
 'perspective': 'First Person',
 'cs_ratio': '30%',
 'gender': 'Female',
 'age': '66+',
 'education_level': 'College',
 'first_language': 'Arabic',
 'second_language': 'English',
 'conversation_type': 'multi-turn',
 'cs_function': 'Expressive',
 'cs_type': 'Extra-sentential / Tag switching',
 'data_generation_result': "['أنا تابعت خبر ترامب عن الفيزا للعمال المهرة، honestly الموضوع معقد شوي.', 'أشوف ناس كثير عندهم آراء مختلفة، you know?', 'على العموم، كل واحد يشوف الموضوع من زاويته، right?', 'الحياة تغيرت كثير في السنوات الأخيرة، believe me.']",
 'fluency_result': '{\'fluency_score\': 8, \'errors\': [{\'description\': "Switching occurs at clause boundaries or after complete phrases, which is generally acceptable. However, in the first sentence, the switch to \'honestly الموضوع معقد شوي.\' is slightly abrupt and could be smoother if integrated more naturally.", \'constraint_violated\': "Slight unnaturalness in discourse-level switching, but no c

## 2. Initial Dataset Exploration

## Parse `data_generation_result`

The `data_generation_result` column is stored as a string representation
of a Python list. We convert it into an actual list of utterances.

In [47]:
# Create the parsing function
def parse_generation_result(value):
    try:
        result = ast.literal_eval(value)

        if isinstance(result, list):
            return result

        return [str(result)]

    except (ValueError, SyntaxError):
        return None

In [48]:
# Test it on one example
raw_text = sample_rows[0]["data_generation_result"]

parsed_text = parse_generation_result(raw_text)

print("Original type:", type(raw_text))
print("Parsed type:", type(parsed_text))
print()
print(parsed_text)

Original type: <class 'str'>
Parsed type: <class 'list'>

['أنا تابعت خبر ترامب عن الفيزا للعمال المهرة، honestly الموضوع معقد شوي.', 'أشوف ناس كثير عندهم آراء مختلفة، you know?', 'على العموم، كل واحد يشوف الموضوع من زاويته، right?', 'الحياة تغيرت كثير في السنوات الأخيرة، believe me.']


In [49]:
# Test parsing on all 1,000 sampled rows
success = 0
failed = 0
empty = 0

for row in sample_rows:
    parsed = parse_generation_result(
        row["data_generation_result"]
    )

    if parsed is None:
        failed += 1
    elif len(parsed) == 0:
        empty += 1
    else:
        success += 1

print("Successfully parsed:", success)
print("Failed:", failed)
print("Empty:", empty)

Successfully parsed: 1000
Failed: 0
Empty: 0


## 3. Build and Inspect the Full Dataset

Apply the validated parser to the full SwitchLingua dataset, retain the fields needed downstream, and inspect data quality before final cleaning.

In [50]:
# Stream the full dataset once and keep only useful columns
from collections import Counter
from datasets import load_dataset

dataset = load_dataset(
    "Shelton1013/SwitchLingua_text",
    split="train",
    streaming=True
)

original_pair_counts = Counter()
retained_pair_counts = Counter()

full_clean_rows = []

for i, row in enumerate(dataset):
    pair = f"{row['first_language']}-{row['second_language']}"

    # Count every original row
    original_pair_counts[pair] += 1

    utterances = parse_generation_result(
        row["data_generation_result"]
    )

    # Skip invalid rows
    if utterances is None or len(utterances) == 0:
        continue

    # Count retained rows
    retained_pair_counts[pair] += 1

    full_clean_rows.append({
        "row_id": i,
        "first_language": row["first_language"],
        "second_language": row["second_language"],
        "conversation_type": row["conversation_type"],
        "cs_type": row["cs_type"],
        "cs_ratio": row["cs_ratio"],
        "score": row["score"],
        "utterances": utterances
    })

print("Clean rows:", len(full_clean_rows))

Clean rows: 234081


In [51]:
# Convert to DataFrame
full_df = pd.DataFrame(full_clean_rows)

print("Shape:", full_df.shape)
print("Columns:", full_df.columns.tolist())

full_df.head()

Shape: (234081, 8)
Columns: ['row_id', 'first_language', 'second_language', 'conversation_type', 'cs_type', 'cs_ratio', 'score', 'utterances']


,row_id,first_language,second_language,conversation_type,cs_type,cs_ratio,score,utterances
0,0,Arabic,English,multi-turn,Extra-sentential / Tag switching,30%,8.700,[أنا تابعت خبر ترامب عن الفيزا للعمال المهرة، ...
1,1,Arabic,English,multi-turn,Intersentential,50%,7.950,[بصراحة، لما سمعت عن استقالة الأمير هاري من ال...
2,2,Arabic,English,single_turn,Intrasentential,70%,8.350,[الهجمات الروسية الأخيرة تسببت في سقوط العديد ...
3,3,Arabic,English,single_turn,Intersentential,50%,8.550,[في المستقبل سأزور الطبيب لمناقشة مشاكلي الصحي...
4,4,Arabic,English,multi-turn,Intersentential,70%,8.675,[زمان كنت أحب أتابع البرامج الموسيقية في التلف...


### Initial Parsing Result

- Original rows: **234,172**
- Retained after parsing: **234,081**
- Removed: **91** (about **0.04%**)

The 91 removed rows could not be parsed into a valid non-empty utterance list.

In [52]:
# Create a language-pair column
comparison_rows = []

for pair in sorted(original_pair_counts):
    original = original_pair_counts[pair]
    retained = retained_pair_counts[pair]

    comparison_rows.append({
        "language_pair": pair,
        "original_count": original,
        "retained_after_cleaning": retained,
        "removed": original - retained
    })

cleaning_summary = pd.DataFrame(comparison_rows)

cleaning_summary

,language_pair,original_count,retained_after_cleaning,removed
0,Arabic-English,8548,8548,0
1,Cantonese-English,119248,119157,91
2,Chinese-English,8458,8458,0
3,French-English,8070,8070,0
4,German-English,6819,6819,0
5,German-French,8511,8511,0
6,Hindi-English,8381,8381,0
7,Italian-English,7979,7979,0
8,Japanese-English,8385,8385,0
9,Korean-English,8266,8266,0


### Parsing Check by Language Pair

All **15 language pairs** remain after parsing. The 91 removed rows are all from Cantonese-English; all other pairs retain their original counts.

### Find records that parsed but are still structurally inconsistent

In [53]:
# Check for rows that do not yet have a simple list[str] utterance format
non_string_rows = full_df[
    full_df["utterances"].apply(
        lambda x: any(not isinstance(item, str) for item in x)
    )
]

print("Rows containing non-string utterance items:", len(non_string_rows))

Rows containing non-string utterance items: 1687


In [54]:
# Inspect a few structured records
non_string_rows[
    ["first_language", "second_language", "utterances"]
].head(10)

,first_language,second_language,utterances
8693,Cantonese,English,"[{'topic': 'business', 'instances': ['最近聽講中國藥品..."
8900,Cantonese,English,"[{'topic': '郭書瑤的最新動態', 'instances': ['你睇唔睇瑤瑤最近..."
8964,Cantonese,English,"[{'topic': 'Math Puzzle Discussion', 'instance..."
8978,Cantonese,English,"[{'topic': 'technology', 'instances': ['有冇睇到呢個..."
8998,Cantonese,English,"[{'topic': 'Government Shutdown Avoidance', 'i..."
9018,Cantonese,English,"[{'topic': 'science', 'instances': ['你睇唔睇到《生活科..."
9159,Cantonese,English,"[{'topic': 'Technology', 'instances': ['我最近睇到一..."
9258,Cantonese,English,"[{'topic': 'sports', 'instances': ['你有冇睇到昨晚嗰場足..."
9300,Cantonese,English,"[{'topic': 'technology', 'instances': ['OpenAI..."
9515,Cantonese,English,"[{'topic': 'science', 'instances': ['最近個新聞真係好誇..."


In [55]:
# Basic data-quality checks before normalization
print("Missing values:")
print(full_df.isnull().sum())

print()
print("Unique row IDs:", full_df["row_id"].nunique())
print("Total parsed rows:", len(full_df))

Missing values:
row_id               0
first_language       0
second_language      0
conversation_type    0
cs_type              0
cs_ratio             0
score                0
utterances           0
dtype: int64

Unique row IDs: 234081
Total parsed rows: 234081


### Data Quality Findings

- **1,687 rows** use structured dictionaries instead of plain-text utterances.
- No required fields are missing.
- Duplicate conversation text is checked **after normalization** so all rows are compared in the same format.

## 4. Normalize Records and Remove Duplicate Conversations

### 4.1 Normalize Utterance Format

Some Cantonese-English records store turns inside dictionaries with an `instances` field. Convert every record to the same `list[str]` format.

In [56]:
# Inspect one structured example
structured_example = non_string_rows.iloc[0]["utterances"]

print(type(structured_example))
print()
print(structured_example)

<class 'list'>

[{'topic': 'business', 'instances': ['最近聽講中國藥品採購好激烈，原廠藥品都冇中標。 Can you believe it?', "係呀，咁樣公立醫院可能會用平價藥，但原廠藥就唔容易搵。 It's a real challenge for people who insist on original brands.", "我覺得呢個情況會改變市場，原廠藥可能會轉去其他渠道。 Maybe they'll focus on private markets instead.", '希望政府會有方法解決，否則一藥難求嘅情況真係唔理想。 We need solutions to ensure accessibility for everyone.']}]


In [57]:
# normalize safely by extracting the instances list
def normalize_utterances(items):
    normalized = []

    for item in items:

        # Normal case: already a string
        if isinstance(item, str):
            normalized.append(item)

        # Structured case: dictionary with an "instances" field
        elif isinstance(item, dict) and "instances" in item:
            instances = item["instances"]

            if isinstance(instances, list):
                normalized.extend(str(x) for x in instances)
            else:
                normalized.append(str(instances))

        # Fallback for unexpected formats
        else:
            normalized.append(str(item))

    return normalized

In [58]:
full_df["utterances"] = (
    full_df["utterances"]
    .apply(normalize_utterances)
)
non_string_rows_after = full_df[
    full_df["utterances"].apply(
        lambda x: any(
            not isinstance(item, str)
            for item in x
        )
    )
]

print(
    "Rows containing non-string utterance items:",
    len(non_string_rows_after)
)
full_df.loc[8693, "utterances"]

Rows containing non-string utterance items: 0


['最近聽講中國藥品採購好激烈，原廠藥品都冇中標。 Can you believe it?',
 "係呀，咁樣公立醫院可能會用平價藥，但原廠藥就唔容易搵。 It's a real challenge for people who insist on original brands.",
 "我覺得呢個情況會改變市場，原廠藥可能會轉去其他渠道。 Maybe they'll focus on private markets instead.",
 '希望政府會有方法解決，否則一藥難求嘅情況真係唔理想。 We need solutions to ensure accessibility for everyone.']

In [59]:
# check for empty lists
empty_utterance_rows = full_df[
    full_df["utterances"].apply(len) == 0
]

print("Rows with empty utterance lists:", len(empty_utterance_rows))

Rows with empty utterance lists: 0


### 4.2 Remove Duplicate Conversation Text

After normalization, identify duplicates using the language pair and full conversation text. Removing them before splitting reduces train/validation/test leakage.

In [60]:
# Create one comparable text string for each conversation
full_df["utterances_text"] = (
    full_df["utterances"]
    .apply(lambda x: " || ".join(x))
)

duplicate_key = [
    "first_language",
    "second_language",
    "utterances_text"
]

# Find all rows that belong to duplicate conversation groups
duplicate_rows = full_df[
    full_df.duplicated(
        subset=duplicate_key,
        keep=False
    )
].copy()

print("Rows involved in duplicate groups:", len(duplicate_rows))

# Check whether identical conversation text has conflicting metadata
if len(duplicate_rows) > 0:
    duplicate_conflicts = (
        duplicate_rows
        .groupby(duplicate_key)
        .agg(
            row_count=("row_id", "size"),
            cs_type_values=("cs_type", "nunique"),
            cs_ratio_values=("cs_ratio", "nunique"),
            score_values=("score", "nunique"),
            conversation_type_values=("conversation_type", "nunique")
        )
        .reset_index()
    )

    print(
        "Duplicate groups with different cs_type:",
        (duplicate_conflicts["cs_type_values"] > 1).sum()
    )
    print(
        "Duplicate groups with different cs_ratio:",
        (duplicate_conflicts["cs_ratio_values"] > 1).sum()
    )
    print(
        "Duplicate groups with different score:",
        (duplicate_conflicts["score_values"] > 1).sum()
    )
    print(
        "Duplicate groups with different conversation_type:",
        (duplicate_conflicts["conversation_type_values"] > 1).sum()
    )

before_dedup = len(full_df)

# Remove duplicate conversation text within the same language pair
full_df = full_df.drop_duplicates(
    subset=duplicate_key,
    keep="first"
).copy()

after_dedup = len(full_df)

print("Rows before text deduplication:", before_dedup)
print("Rows after text deduplication:", after_dedup)
print("Text duplicates removed:", before_dedup - after_dedup)

# Verify that no duplicate conversation text remains
remaining_duplicates = full_df.duplicated(
    subset=duplicate_key
).sum()

print("Remaining text duplicates:", remaining_duplicates)

Rows involved in duplicate groups: 1042
Duplicate groups with different cs_type: 32
Duplicate groups with different cs_ratio: 41
Duplicate groups with different score: 56
Duplicate groups with different conversation_type: 13
Rows before text deduplication: 234081
Rows after text deduplication: 233100
Text duplicates removed: 981
Remaining text duplicates: 0


We found repeated conversation text with differing metadata, including cs_type. This suggests that some metadata is associated with separate generation records rather than uniquely determined by the final text. Since our downstream targets are derived from the actual language sequence, we deduplicate by conversation text to prevent identical linguistic examples from appearing across train and test sets.

In [62]:
full_df = full_df.drop(columns=["utterances_text"])

In [63]:
full_df["language_pair"] = (
    full_df["first_language"]
    + "-"
    + full_df["second_language"]
)

In [64]:
# Count rows remaining after normalization and deduplication
final_pair_counts = (
    full_df["language_pair"]
    .value_counts()
    .rename_axis("language_pair")
    .reset_index(name="final_count")
)

# Combine with the earlier cleaning summary
pair_summary = cleaning_summary.merge(
    final_pair_counts,
    on="language_pair",
    how="left"
)

pair_summary["removed_during_initial_cleaning"] = (
    pair_summary["original_count"]
    - pair_summary["retained_after_cleaning"]
)

pair_summary["removed_as_duplicates"] = (
    pair_summary["retained_after_cleaning"]
    - pair_summary["final_count"]
)

pair_summary["total_removed"] = (
    pair_summary["original_count"]
    - pair_summary["final_count"]
)

pair_summary = pair_summary[
    [
        "language_pair",
        "original_count",
        "retained_after_cleaning",
        "total_removed",
        "final_count"
    ]
]

pair_summary = pair_summary.sort_values(
    "language_pair"
).reset_index(drop=True)

pair_summary

,language_pair,original_count,retained_after_cleaning,total_removed,final_count
0,Arabic-English,8548,8548,4,8544
1,Cantonese-English,119248,119157,1045,118203
2,Chinese-English,8458,8458,1,8457
3,French-English,8070,8070,3,8067
4,German-English,6819,6819,5,6814
5,German-French,8511,8511,3,8508
6,Hindi-English,8381,8381,0,8381
7,Italian-English,7979,7979,1,7978
8,Japanese-English,8385,8385,0,8385
9,Korean-English,8266,8266,0,8266


## 5. Train / Validation / Test Split

We split the cleaned dataset at the original row level before expanding
individual utterances. This prevents utterances from the same conversation
from appearing in multiple splits.

We use an 80/10/10 split stratified by language pair.

In [65]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    full_df,
    test_size=0.20,
    random_state=42,
    stratify=full_df["language_pair"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["language_pair"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 186480
Validation: 23310
Test: 23310


In [66]:
# verify there is no overlap
train_ids = set(train_df["row_id"])
val_ids = set(val_df["row_id"])
test_ids = set(test_df["row_id"])

print("Train/Val overlap:", len(train_ids & val_ids))
print("Train/Test overlap:", len(train_ids & test_ids))
print("Val/Test overlap:", len(val_ids & test_ids))

Train/Val overlap: 0
Train/Test overlap: 0
Val/Test overlap: 0


In [67]:
# create a clean split-summary table
split_summary = pd.concat(
    [
        train_df["language_pair"].value_counts().rename("train"),
        val_df["language_pair"].value_counts().rename("validation"),
        test_df["language_pair"].value_counts().rename("test")
    ],
    axis=1
).fillna(0).astype(int)

split_summary["total"] = (
    split_summary["train"]
    + split_summary["validation"]
    + split_summary["test"]
)

split_summary = (
    split_summary
    .reset_index()
    .rename(columns={"index": "language_pair"})
    .sort_values("language_pair")
    .reset_index(drop=True)
)

split_summary

,language_pair,train,validation,test,total
0,Arabic-English,6835,855,854,8544
1,Cantonese-English,94562,11821,11820,118203
2,Chinese-English,6765,846,846,8457
3,French-English,6454,807,806,8067
4,German-English,5451,682,681,6814
5,German-French,6806,851,851,8508
6,Hindi-English,6705,838,838,8381
7,Italian-English,6382,798,798,7978
8,Japanese-English,6708,838,839,8385
9,Korean-English,6613,826,827,8266


## 6. Save Processed Data

In [68]:
import os

os.makedirs("data/processed", exist_ok=True)

train_df.to_parquet(
    "data/processed/train.parquet",
    index=False
)

val_df.to_parquet(
    "data/processed/validation.parquet",
    index=False
)

test_df.to_parquet(
    "data/processed/test.parquet",
    index=False
)

print("Saved train, validation, and test splits.")

Saved train, validation, and test splits.


In [69]:
# Create language-pair distribution for team handoff
pair_distribution = (
    full_df["language_pair"]
    .value_counts()
    .rename_axis("language_pair")
    .reset_index(name="count")
)

pair_distribution["percentage"] = (
    pair_distribution["count"]
    / pair_distribution["count"].sum()
    * 100
).round(2)

# Save summary tables
split_summary.to_csv(
    "data/processed/split_summary.csv",
    index=False
)

pair_summary.to_csv(
    "data/processed/cleaning_summary.csv",
    index=False
)

pair_distribution.to_csv(
    "data/processed/language_pair_distribution.csv",
    index=False
)

print("Saved summary tables.")

Saved summary tables.


## 7. Dataset Statistics

Summarize the final cleaned dataset and check whether major metadata distributions are balanced.

### 7.1 Overall Dataset Size

In [70]:
dataset_summary = pd.DataFrame({
    "Statistic": [
        "Final usable rows",
        "Number of language pairs",
        "Training rows",
        "Validation rows",
        "Test rows"
    ],
    "Value": [
        len(full_df),
        full_df["language_pair"].nunique(),
        len(train_df),
        len(val_df),
        len(test_df)
    ]
})

dataset_summary

,Statistic,Value
0,Final usable rows,233100
1,Number of language pairs,15
2,Training rows,186480
3,Validation rows,23310
4,Test rows,23310


The final dataset is split approximately **80/10/10** into train, validation, and test sets, stratified by language pair.

### 7.2 Language-Pair Distribution

In [71]:
pair_distribution

,language_pair,count,percentage
0,Cantonese-English,118203,50.71
1,Arabic-English,8544,3.67
2,Philippines-English,8537,3.66
3,German-French,8508,3.65
4,Chinese-English,8457,3.63
5,Vietnamese-English,8456,3.63
6,Malay-English,8386,3.60
7,Japanese-English,8385,3.60
8,Hindi-English,8381,3.60
9,Korean-English,8266,3.55


Language-pair sizes are highly imbalanced: **Cantonese-English makes up about half of the dataset**, while most other pairs are much smaller. Pair-specific evaluation is therefore important for the universality analysis.

### 7.3 Overall Code-Switch Ratio

In [72]:
cs_ratio_counts = (
    full_df["cs_ratio"]
    .value_counts()
    .rename_axis("cs_ratio")
    .reset_index(name="count")
)

cs_ratio_counts

,cs_ratio,count
0,30%,77951
1,50%,77719
2,70%,77430


The 30%, 50%, and 70% `cs_ratio` groups are nearly evenly represented overall.

### 7.4 Code-Switch Ratio by Language Pair

In [73]:
cs_ratio_by_pair = pd.crosstab(
    full_df["language_pair"],
    full_df["cs_ratio"]
)

cs_ratio_by_pair

cs_ratio,30%,50%,70%
language_pair,,,
Arabic-English,2907,2878,2759
Cantonese-English,39492,39274,39437
Chinese-English,2813,2909,2735
French-English,2678,2703,2686
German-English,2291,2316,2207
German-French,2891,2868,2749
Hindi-English,2750,2878,2753
Italian-English,2649,2688,2641
Japanese-English,2980,2636,2769


In [74]:
cs_ratio_pct_by_pair = (
    pd.crosstab(
        full_df["language_pair"],
        full_df["cs_ratio"],
        normalize="index"
    ) * 100
).round(2)

cs_ratio_pct_by_pair

cs_ratio,30%,50%,70%
language_pair,,,
Arabic-English,34.02,33.68,32.29
Cantonese-English,33.41,33.23,33.36
Chinese-English,33.26,34.40,32.34
French-English,33.20,33.51,33.30
German-English,33.62,33.99,32.39
German-French,33.98,33.71,32.31
Hindi-English,32.81,34.34,32.85
Italian-English,33.20,33.69,33.10
Japanese-English,35.54,31.44,33.02


Within most language pairs, the 30%, 50%, and 70% ratios are also close to evenly distributed. This reduces `cs_ratio` as a likely source of pair-level performance differences.

### 7.5 Conversation-Type Distribution

In [75]:
conversation_type_summary = (
    full_df["conversation_type"]
    .value_counts()
    .rename_axis("conversation_type")
    .reset_index(name="count")
)

conversation_type_summary["percentage"] = (
    conversation_type_summary["count"]
    / conversation_type_summary["count"].sum()
    * 100
).round(2)

conversation_type_summary

,conversation_type,count,percentage
0,single_turn,116699,50.06
1,multi-turn,116401,49.94


Single-turn and multi-turn examples are approximately evenly represented.

### 7.6 Code-Switch Type Distribution

In [76]:
cs_type_summary = (
    full_df["cs_type"]
    .value_counts()
    .rename_axis("cs_type")
    .reset_index(name="count")
)

cs_type_summary["percentage"] = (
    cs_type_summary["count"]
    / cs_type_summary["count"].sum()
    * 100
).round(2)

cs_type_summary

,cs_type,count,percentage
0,Extra-sentential / Tag switching,77939,33.44
1,Intersentential,77917,33.43
2,Intrasentential,77244,33.14


The three code-switch types are approximately balanced, with each contributing about one-third of the dataset.

### 7.7 Utterances per Conversation

In [77]:
full_df["num_utterances"] = (
    full_df["utterances"].apply(len)
)

full_df["num_utterances"].describe()

count    233100.000000
mean          4.169464
std           2.142580
min           1.000000
25%           3.000000
50%           4.000000
75%           5.000000
max          31.000000
Name: num_utterances, dtype: float64

In [78]:
utterance_count_summary = pd.DataFrame({
    "Statistic": [
        "Mean",
        "Median",
        "Minimum",
        "Maximum"
    ],
    "Utterances per row": [
        full_df["num_utterances"].mean(),
        full_df["num_utterances"].median(),
        full_df["num_utterances"].min(),
        full_df["num_utterances"].max()
    ]
})

utterance_count_summary

,Statistic,Utterances per row
0,Mean,4.169464
1,Median,4.000000
2,Minimum,1.000000
3,Maximum,31.000000


Most conversations contain about **four utterances**, although a small number are much longer. Splitting at the original conversation level keeps all utterances from one conversation in the same data split.

### 7.8 Utterance Length

In [79]:
# Expand each conversation into individual utterances for length statistics
utterance_df = (
    full_df[
        ["row_id", "language_pair", "utterances"]
    ]
    .explode("utterances")
    .rename(columns={"utterances": "text"})
    .reset_index(drop=True)
)

# Character length is used because whitespace-based word counts
# are not directly comparable across all languages.
utterance_df["char_length"] = utterance_df["text"].str.len()

print("Total utterances:", len(utterance_df))

utterance_df["char_length"].describe()

Total utterances: 971902


count    971902.000000
mean         61.251131
std          47.087859
min           0.000000
25%          27.000000
50%          50.000000
75%          83.000000
max       14325.000000
Name: char_length, dtype: float64

In [80]:
utterance_length_by_pair = (
    utterance_df
    .groupby("language_pair")["char_length"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        min="min",
        max="max"
    )
    .round(2)
    .reset_index()
)

utterance_length_by_pair

,language_pair,count,mean,median,min,max
0,Arabic-English,26787,85.57,81.0,14,542
1,Cantonese-English,616817,40.18,32.0,0,14325
2,Chinese-English,29558,67.21,63.0,7,528
3,French-English,23560,110.69,105.0,22,493
4,German-English,19578,110.38,106.0,25,549
5,German-French,23592,113.25,108.0,18,458
6,Hindi-English,26691,98.78,94.0,19,489
7,Italian-English,24164,111.89,104.0,26,529
8,Japanese-English,27885,74.31,72.0,12,538
9,Korean-English,25383,71.31,68.0,12,448


Character count is used as a preliminary cross-language length measure because whitespace-based word counts are not comparable across all languages. Subword and switched-segment lengths will be computed after tokenization and Language ID alignment.

In [81]:
import pandas as pd

train = pd.read_parquet("data/processed/train.parquet")
val = pd.read_parquet("data/processed/validation.parquet")
test = pd.read_parquet("data/processed/test.parquet")

print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))
print("Total:", len(train) + len(val) + len(test))

Train: 186480
Validation: 23310
Test: 23310
Total: 233100
